In [2]:
import requests
from bs4 import BeautifulSoup
import time
import random
import pandas as pd
from tqdm import tqdm
import os
from urllib.parse import urljoin

# 设置请求头
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Referer': 'https://pubmed.ncbi.nlm.nih.gov/',
    'DNT': '1',
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'same-origin',
    'Sec-Fetch-User': '?1',
}

# 网页URL
BASE_URL = "https://pubmed.ncbi.nlm.nih.gov"
SEARCH_URL = "https://pubmed.ncbi.nlm.nih.gov/?term=bee+pollination"  #1.关键词为bee pollination时筛选方式为most recent 2.关键词为bee pollination时筛选方式为best match 3.关键词为honeybee时筛选方式为most recent 4.关键词为honeybee时筛选方式为best match


# 结果
CSV_FILE = "pubmed_articles.csv"

# 变量
all_articles = []
article_count = 0
start_page = 1
max_pages = 2000  # 最大爬取总页数

# 检查是否有已保存的文件，如果有则从上次中断处继续
if os.path.exists(CSV_FILE):
    try:
        existing_data = pd.read_csv(CSV_FILE)
        article_count = len(existing_data)
        start_page = (article_count // 10) + 1
        all_articles = existing_data.to_dict('records')
        print(f"检测到已有数据，从第 {start_page} 页继续爬取，已爬取 {article_count} 条记录")
    except Exception as e:
        print(f"读取已有文件出错: {e}，将从头开始爬取")


def get_full_abstract(article_url):
    """获取文章详情页的完整摘要"""
    try:
        # 随机延时，避免请求过于频繁
        time.sleep(random.uniform(2, 5))

        response = requests.get(article_url, headers=HEADERS)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        # 查找摘要容器 - PubMed的摘要通常在class为"abstract-content"的div中
        abstract_sections = soup.find_all('div', class_='abstract-content')

        if not abstract_sections:
            # 尝试其他可能的摘要容器
            abstract_sections = soup.find_all('div', class_='abstract')

        if abstract_sections:
            # 合并所有摘要段落
            full_abstract = "\n".join(
                section.get_text(strip=True)
                for section in abstract_sections
            )
            return full_abstract
        else:
            return "无完整摘要"

    except Exception as e:
        print(f"获取完整摘要失败: {article_url} - {str(e)}")
        return "获取摘要失败"


# 创建进度条
with tqdm(total=max_pages * 10, initial=article_count, desc="爬取进度") as pbar:
    # 遍历所有页面
    for page_num in range(start_page, max_pages + 1):
        # 构建页面URL
        page_url = f"{SEARCH_URL}&page={page_num}"

        # 发送请求
        try:
            response = requests.get(page_url, headers=HEADERS)
            response.raise_for_status()  # 检查请求是否成功
        except Exception as e:
            print(f"请求第 {page_num} 页失败: {e}")
            time.sleep(10)  # 等待10秒后重试
            continue

        # 解析页面
        soup = BeautifulSoup(response.text, 'html.parser')

        # 查找所有文章容器
        articles = soup.find_all('div', class_='docsum-wrap')

        # 如果没有找到文章，可能是页面结构变化或到达最后一页
        if not articles:
            print(f"第 {page_num} 页没有找到文章，可能是页面结构变化或到达最后一页")
            break

        # 处理每篇文章
        for article in articles:
            try:
                # 提取标题和链接
                title_element = article.find('a', class_='docsum-title')
                if title_element:
                    title = title_element.get_text(strip=True)
                    link = urljoin(BASE_URL, title_element['href'])
                else:
                    title = "无标题"
                    link = "无链接"

                # 获取完整摘要
                full_abstract = get_full_abstract(link) if link != "无链接" else "无链接"

                # 添加到列表
                article_count += 1
                all_articles.append({
                    '序号': article_count,
                    '标题': title,
                    '链接': link,
                    '完整摘要': full_abstract
                })

                # 通知用户
                print(f"已爬取第 {article_count} 条: {title[:50]}...")

                # 更新进度条
                pbar.update(1)

                # 每爬取5条保存一次（减少保存频率）
                if article_count % 5 == 0:
                    df = pd.DataFrame(all_articles)
                    df.to_csv(CSV_FILE, index=False, encoding='utf-8-sig')
                    print(f"已保存 {article_count} 条数据到 {CSV_FILE}")

            except Exception as e:
                print(f"处理文章时出错: {e}")
                continue

        # 随机等待5-10秒，避免请求过于频繁
        wait_time = random.uniform(5, 10)
        time.sleep(wait_time)

        # 检查是否还有下一页
        next_button = soup.find('button', class_='next-page-btn')
        if not next_button or 'disabled' in next_button.get('class', []):
            print("已到达最后一页")
            break

# 保存最终结果
if all_articles:
    df = pd.DataFrame(all_articles)
    df.to_csv(CSV_FILE, index=False, encoding='utf-8-sig')
    print(f"爬取完成！共爬取 {len(all_articles)} 条数据，结果已保存到 {CSV_FILE}")
else:
    print("没有爬取到任何数据")


检测到已有数据，从第 1001 页继续爬取，已爬取 10000 条记录


爬取进度:  50%|█████     | 10000/20000 [00:02<?, ?it/s]

第 1001 页没有找到文章，可能是页面结构变化或到达最后一页
爬取完成！共爬取 10000 条数据，结果已保存到 pubmed_articles.csv
